In [ ]:
# Initial EDA for Smart Load Shedding Optimizer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium  # For maps
from pathlib import Path
import os

# Set plot style
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 8)

# Create outputs directory if it doesn't exist
os.makedirs('../outputs', exist_ok=True)

# Load processed data
DATA_DIR = Path('../data/processed')
print("Loading data from:", DATA_DIR)

# Load files if they exist, otherwise create empty DataFrames with a message
def safe_load_parquet(filepath, name):
    if filepath.exists():
        df = pd.read_parquet(filepath)
        print(f"✅ Loaded {name} data: {df.shape[0]} rows, {df.shape[1]} columns")
        return df
    else:
        print(f"⚠️ Warning: {filepath} not found. Using empty DataFrame for {name}.")
        return pd.DataFrame()

demand_df = safe_load_parquet(DATA_DIR / 'demand.parquet', 'demand')
outage_df = safe_load_parquet(DATA_DIR / 'outages.parquet', 'outage')
weather_df = safe_load_parquet(DATA_DIR / 'weather.parquet', 'weather')
grid_df = safe_load_parquet(DATA_DIR / 'grid_infra.parquet', 'grid infrastructure')
critical_df = safe_load_parquet(DATA_DIR / 'critical_infra.parquet', 'critical infrastructure')

# Check if we have actual data to analyze
has_demand_data = not demand_df.empty
has_outage_data = not outage_df.empty
has_weather_data = not weather_df.empty
has_grid_data = not grid_df.empty
has_critical_data = not critical_df.empty

print("\n--- Data Availability Summary ---")
print(f"Demand data: {'Available' if has_demand_data else 'Missing'}")
print(f"Outage data: {'Available' if has_outage_data else 'Missing'}")
print(f"Weather data: {'Available' if has_weather_data else 'Missing'}")
print(f"Grid data: {'Available' if has_grid_data else 'Missing'}")
print(f"Critical infrastructure data: {'Available' if has_critical_data else 'Missing'}")
print("-------------------------------\n")

# Section 1: Demand Data EDA
if has_demand_data:
    print("\n==== DEMAND DATA ANALYSIS ====")
    print("Demand Data Shape:", demand_df.shape)
    print("\nSample data:")
    display(demand_df.head())
    
    print("\nData types:")
    display(demand_df.dtypes)
    
    print("\nSummary statistics:")
    display(demand_df.describe())
    
    print("\nChecking for missing values:")
    display(demand_df.isnull().sum())
    
    # Make sure timestamp is datetime type
    if 'timestamp' in demand_df.columns:
        if not pd.api.types.is_datetime64_any_dtype(demand_df['timestamp']):
            print("Converting timestamp to datetime...")
            demand_df['timestamp'] = pd.to_datetime(demand_df['timestamp'])
            
        # Time series of demand
        plt.figure(figsize=(14, 6))
        # Group by date and plot average demand
        if 'demand_mw' in demand_df.columns:
            daily_demand = demand_df.groupby(demand_df['timestamp'].dt.date)['demand_mw'].mean()
            daily_demand.plot(title='Average Daily Electricity Demand')
            plt.ylabel('Demand (MW)')
            plt.grid(True)
            plt.tight_layout()
            plt.savefig('../outputs/daily_demand.png')
            print("Saved plot to ../outputs/daily_demand.png")
            
            # Monthly patterns
            demand_df['month'] = demand_df['timestamp'].dt.month
            demand_df['year'] = demand_df['timestamp'].dt.year
            demand_df['day_of_week'] = demand_df['timestamp'].dt.dayofweek
            
            if 'hour' not in demand_df.columns:
                demand_df['hour'] = demand_df['timestamp'].dt.hour
    
            plt.figure(figsize=(14, 6))
            sns.boxplot(x='month', y='demand_mw', data=demand_df)
            plt.title('Monthly Demand Distribution')
            plt.xlabel('Month')
            plt.ylabel('Demand (MW)')
            plt.savefig('../outputs/monthly_demand.png')
            print("Saved plot to ../outputs/monthly_demand.png")
            
            # Day of week patterns
            plt.figure(figsize=(12, 6))
            sns.boxplot(x='day_of_week', y='demand_mw', data=demand_df)
            plt.title('Demand by Day of Week')
            plt.xlabel('Day (0=Monday, 6=Sunday)')
            plt.ylabel('Demand (MW)')
            plt.savefig('../outputs/day_of_week_demand.png')
            print("Saved plot to ../outputs/day_of_week_demand.png")
            
            # Hour of day patterns
            plt.figure(figsize=(14, 6))
            hourly_demand = demand_df.groupby('hour')['demand_mw'].mean()
            hourly_demand.plot(kind='line', marker='o')
            plt.title('Average Demand by Hour of Day')
            plt.xlabel('Hour')
            plt.ylabel('Average Demand (MW)')
            plt.xticks(range(0, 24))
            plt.grid(True)
            plt.savefig('../outputs/hourly_demand.png')
            print("Saved plot to ../outputs/hourly_demand.png")
        else:
            print("Warning: 'demand_mw' column not found in demand data")

# Section 2: Outage Data EDA
if has_outage_data:
    print("\n==== OUTAGE DATA ANALYSIS ====")
    print("Outage Data Shape:", outage_df.shape)
    print("\nSample data:")
    display(outage_df.head())
    
    print("\nData types:")
    display(outage_df.dtypes)
    
    print("\nSummary statistics:")
    display(outage_df.describe(include='all'))
    
    print("\nChecking for missing values:")
    display(outage_df.isnull().sum())
    
    # Outage frequency by region
    if 'region_id' in outage_df.columns:
        plt.figure(figsize=(14, 8))
        outage_counts = outage_df['region_id'].value_counts().sort_values(ascending=False)
        outage_counts.plot(kind='bar', title='Outage Frequency by Region')
        plt.ylabel('Number of Outages')
        plt.tight_layout()
        plt.savefig('../outputs/outage_by_region.png')
        print("Saved plot to ../outputs/outage_by_region.png")
    
    # Outage duration analysis
    if 'start_time' in outage_df.columns and 'end_time' in outage_df.columns:
        outage_df['duration_hours'] = (outage_df['end_time'] - outage_df['start_time']).dt.total_seconds() / 3600
        
        plt.figure(figsize=(10, 6))
        sns.histplot(outage_df['duration_hours'], bins=20)
        plt.title('Distribution of Outage Durations')
        plt.xlabel('Duration (hours)')
        plt.savefig('../outputs/outage_duration_dist.png')
        print("Saved plot to ../outputs/outage_duration_dist.png")
        
        # Average outage duration by region
        plt.figure(figsize=(14, 8))
        region_duration = outage_df.groupby('region_id')['duration_hours'].mean().sort_values(ascending=False)
        region_duration.plot(kind='bar', title='Average Outage Duration by Region')
        plt.ylabel('Average Duration (hours)')
        plt.tight_layout()
        plt.savefig('../outputs/avg_outage_duration.png')
        print("Saved plot to ../outputs/avg_outage_duration.png")

# Section 3: Weather Data EDA
if has_weather_data:
    print("\n==== WEATHER DATA ANALYSIS ====")
    print("Weather Data Shape:", weather_df.shape)
    print("\nSample data:")
    display(weather_df.head())
    
    print("\nData types:")
    display(weather_df.dtypes)
    
    print("\nSummary statistics:")
    display(weather_df.describe())
    
    print("\nChecking for missing values:")
    display(weather_df.isnull().sum())
    
    # Weather variables distributions
    if 'temperature' in weather_df.columns:
        plt.figure(figsize=(10, 6))
        sns.histplot(weather_df['temperature'], kde=True)
        plt.title('Temperature Distribution')
        plt.savefig('../outputs/temp_distribution.png')
        print("Saved plot to ../outputs/temp_distribution.png")

# Section 4: Weather-Demand Correlation
if has_demand_data and has_weather_data and 'region_id' in demand_df.columns and 'region_id' in weather_df.columns:
    print("\n==== WEATHER-DEMAND CORRELATION ====")
    
    # Ensure timestamp is datetime type in both DataFrames
    if 'timestamp' in weather_df.columns:
        if not pd.api.types.is_datetime64_any_dtype(weather_df['timestamp']):
            weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])
    
    # Merge demand and weather data
    try:
        merged_df = pd.merge(
            demand_df, 
            weather_df,
            on=['timestamp', 'region_id'],
            how='inner'
        )
        
        print("\nMerged Demand-Weather Data Shape:", merged_df.shape)
        print("\nSample merged data:")
        display(merged_df.head())
        
        # Correlation heatmap
        if 'demand_mw' in merged_df.columns and 'temperature' in merged_df.columns:
            plt.figure(figsize=(10, 8))
            numeric_cols = merged_df.select_dtypes(include=[np.number]).columns
            corr = merged_df[numeric_cols].corr()
            sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
            plt.title('Correlation Matrix: Demand vs. Weather')
            plt.tight_layout()
            plt.savefig('../outputs/demand_weather_correlation.png')
            print("Saved plot to ../outputs/demand_weather_correlation.png")
            
            # Scatter plot: Temperature vs. Demand
            plt.figure(figsize=(10, 6))
            sns.scatterplot(x='temperature', y='demand_mw', data=merged_df, alpha=0.5)
            plt.title('Temperature vs. Electricity Demand')
            plt.xlabel('Temperature (°C)')
            plt.ylabel('Demand (MW)')
            plt.savefig('../outputs/temp_vs_demand.png')
            print("Saved plot to ../outputs/temp_vs_demand.png")
            
            # Define extreme weather thresholds if needed
            if merged_df['temperature'].max() > 30:  # Only if we have hot temperatures
                threshold = 30  # Adjust threshold based on your region
                merged_df['extreme_temp'] = merged_df['temperature'] > threshold
                
                # Compare demand during extreme vs. normal conditions
                plt.figure(figsize=(10, 6))
                sns.boxplot(x='extreme_temp', y='demand_mw', data=merged_df)
                plt.title(f'Impact of High Temperatures (>{threshold}°C) on Demand')
                plt.xlabel('High Temperature Event')
                plt.ylabel('Demand (MW)')
                plt.savefig('../outputs/extreme_temp_impact.png')
                print("Saved plot to ../outputs/extreme_temp_impact.png")
    except Exception as e:
        print(f"Error merging demand and weather data: {e}")

# Section 5: Grid Infrastructure Visualization
if has_grid_data:
    print("\n==== GRID INFRASTRUCTURE ANALYSIS ====")
    print("Grid Data Shape:", grid_df.shape)
    print("\nSample data:")
    display(grid_df.head())
    
    # Basic map visualization (if coordinates available)
    if 'latitude' in grid_df.columns and 'longitude' in grid_df.columns:
        print("Creating infrastructure map...")
        try:
            # Calculate center of map
            center_lat = grid_df['latitude'].mean()
            center_lon = grid_df['longitude'].mean()
            
            m = folium.Map(location=[center_lat, center_lon], zoom_start=5)
            
            # Add grid infrastructure points
            for _, row in grid_df.iterrows():
                folium.Marker(
                    [row['latitude'], row['longitude']],
                    popup=f"Type: {row.get('infra_type', 'Grid')}<br>Capacity: {row.get('capacity', 'N/A')}"
                ).add_to(m)
            
            # Save map as HTML
            map_path = '../outputs/grid_infrastructure_map.html'
            m.save(map_path)
            print(f"Map saved to {map_path}")
        except Exception as e:
            print(f"Error creating map: {e}")

# Section 6: Missing Data Analysis
print("\n==== MISSING DATA ANALYSIS ====")
datasets = {}
if has_demand_data: datasets['Demand'] = demand_df
if has_weather_data: datasets['Weather'] = weather_df
if has_outage_data: datasets['Outages'] = outage_df
if has_grid_data: datasets['Grid'] = grid_df
if has_critical_data: datasets['Critical'] = critical_df

if datasets:
    plt.figure(figsize=(12, 6))
    missing_data = {name: df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100 
                  for name, df in datasets.items()}
    
    plt.bar(missing_data.keys(), missing_data.values())
    plt.title('Missing Data Percentage by Dataset')
    plt.ylabel('Missing Values (%)')
    plt.savefig('../outputs/missing_data_percentage.png')
    print("Saved plot to ../outputs/missing_data_percentage.png")

print("\n==== EDA COMPLETE ====")